# Gayaku garment classifier — YOLOv8 classification on Colab

This notebook trains the app's garment classifier with **Ultralytics YOLOv8** (`yolov8n-cls`) and exports the one file the Flutter app loads: a LiteRT `.tflite`. It is the YOLO counterpart of `ml_workspace/kaggle/fashion_classifier.ipynb` (MobileNetV3-small) and it replaces that model — see `docs/adr/0006-yolov8-classifier-agpl.md` for why, and for the licence consequence (step k).

Run it top to bottom on a free Colab GPU (`Runtime > Change runtime type > T4 GPU`). It reuses this repository's own scripts (`ml_workspace/scripts/prepare_data.py`, `ml_workspace/bootstrap/fetch_dataset.py`, `ml_workspace/labels.json`) instead of reimplementing them, writes everything under `ml_workspace/yolo/`, and ends by zipping what you need to download.

| step | what it does |
| --- | --- |
| a | check the runtime (GPU, versions) and install `ultralytics` + `kaggle` |
| b | clone the repository into `/content/gayaku` |
| c | read a Kaggle API token, if you have one |
| d | choose the training data: Kaggle catalogue or the credential-free CC0 set |
| e | build the ultralytics ImageFolder out of symlinks |
| f | train `yolov8n-cls` at 224 px |
| g | validate: top-1/top-5, per-class accuracy, confusion matrix |
| h | export to `.tflite`, then verify the contract and the conversion |
| i | write `classes.json` and `report.md` |
| j | zip the artifacts and download them |
| k | what the AGPL-3.0 licence means for this build |

**Prerequisites** (the checklist with disk/time expectations is `ml_workspace/yolo/README.md`): a Kaggle account with an API token **or** the CC0 path (`DATASET = "cc0-bootstrap"`, no credentials). Which classes exist is decided by `keep: true` entries in `ml_workspace/labels.json` — each one needs a Category and an Indonesian name before it can reach the app.

The app's frozen contract, enforced at load by `flutter_app/lib/logic/scan_pipeline.dart::checkContracts`:

* **input** float32 (`float16` tolerated), NHWC `[1,224,224,3]` or NCHW `[1,3,224,224]`, RGB in `[0,1]` — any normalisation is baked into the graph;
* **output** one tensor `[1,N]` float32, `N` = the number of labels in the shipped class list, and the values must be a probability distribution;
* **class order** = the order of `labels` in `classes.json`, which `scripts/gen_dart_labels.py` turns into `flutter_app/lib/logic/garment_labels.dart`.

Step h asserts every line of that on the exported file, with an independent TFLite runtime.

## a. Check the runtime, install the libraries

Two libraries, both installed by the cell below:

* **ultralytics** — YOLOv8 training, validation and the LiteRT (`.tflite`) export;
* **kaggle** — the downloader that `ml_workspace/scripts/prepare_data.py` shells out to.

The rest of the cell is a plain environment report, because almost every failure later in this notebook looks the same from the outside: wrong runtime. We print the Python, PyTorch and ultralytics versions (ultralytics changes its API between releases, so knowing the exact version is how you tell "our bug" from "their change"), whether CUDA is visible, and which GPU it is. `DEVICE` is set here and used by every later cell.

No GPU is not fatal — training runs on CPU — but the 80-class catalogue takes many hours there, so the cell says so instead of pretending. It also probes for the TFLite runtime that step h uses to verify the export: TensorFlow ships with Colab, and `ai_edge_litert` arrives with the exporter, so at least one is always present.

In [ ]:
# Step a (1/11) — look at the runtime, then install the two libraries this notebook drives.
#
#   ultralytics -> YOLOv8 classification training, validation, LiteRT (.tflite) export
#   kaggle      -> the dataset downloader that ml_workspace/scripts/prepare_data.py shells out to
#
# `!pip` installs into the environment the kernel runs in; the `import ultralytics` below fails
# loudly if that ever stops being true.
!pip install -q ultralytics kaggle

import importlib.util
import sys

import torch
import ultralytics

print(f"python        {sys.version.split()[0]}")
print(f"torch         {torch.__version__}")
print(f"ultralytics   {ultralytics.__version__}")

GPU = torch.cuda.is_available()
if GPU:
    DEVICE = 0
    print(f"cuda          {torch.version.cuda}")
    print(f"gpu           {torch.cuda.get_device_name(0)}")
else:
    DEVICE = "cpu"
    print("cuda          NOT AVAILABLE")
    print("              Runtime > Change runtime type > Hardware accelerator: T4 GPU")
    print("              Training still runs on CPU, but the catalogue needs many hours there.")

# Step h verifies the exported .tflite with an independent runtime. Colab ships TensorFlow;
# ai_edge_litert comes with the LiteRT exporter, so one of the two is always importable.
for module in ("tensorflow", "ai_edge_litert"):
    print(f"{module:<14}{'available' if importlib.util.find_spec(module) else 'not installed yet'}")

## b. Clone the repository

The notebook drives this repository's scripts rather than a copy of their logic, so step 1 is a clone. `REPO_URL` and `BRANCH` are variables: point them at your own fork if you have one, and at the branch you are actually testing.

`--depth 1` keeps the clone small (history is irrelevant here). We then `chdir` into the clone, because every command in `ml_workspace/README.md` is documented as run from the repository root, and the paths a script prints are relative to it.

The cell also defines the three directories everything else writes to, all under `ml_workspace/yolo/`: `dataset/` (the ultralytics ImageFolder, symlinks only), `runs/` (training and validation output), `artifacts/` (the files you download at the end). The commit hash is captured for `report.md`, so a set of numbers can always be traced back to the exact code that produced it.

In [ ]:
# Step b (2/11) — clone the repository, so the notebook reuses our own scripts instead of a copy.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kmilaz/gayaku.git"   # your fork goes here
BRANCH = "main"                                    # the branch under test
CLONE_DIR = Path("/content/gayaku")

if (CLONE_DIR / ".git").is_dir():
    print(f"{CLONE_DIR} is already cloned; reusing it (delete the folder to start clean)")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(CLONE_DIR)],
        check=True,
    )

REPO = CLONE_DIR.resolve()
os.chdir(REPO)  # every command in ml_workspace/README.md runs from the repository root

if not (REPO / "ml_workspace" / "scripts" / "prepare_data.py").is_file():
    raise SystemExit(f"{REPO} does not look like the gayaku repository")

GIT_SHA = subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

# Everything this notebook writes lives under ml_workspace/yolo/ -- see ml_workspace/yolo/README.md.
YOLO_DIR = REPO / "ml_workspace" / "yolo"
DATASET_DIR = YOLO_DIR / "dataset"      # ultralytics ImageFolder: dataset/{train,val,test}/<Label>/
RUNS_DIR = YOLO_DIR / "runs"            # training and validation runs
ARTIFACTS_DIR = YOLO_DIR / "artifacts"  # the files you download at the end
for path in (DATASET_DIR, RUNS_DIR, ARTIFACTS_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(f"repo          {REPO}")
print(f"commit        {GIT_SHA}")
print(f"artifacts     {ARTIFACTS_DIR}")

## c. Kaggle API token (optional)

The catalogue dataset lives on Kaggle, and `prepare_data.py` uses the `kaggle` CLI. Kaggle now hands out **two shapes of credential** and the CLI accepts both:

1. `kaggle.json` — the classic `{"username": ..., "key": ...}` file, read from `~/.kaggle/kaggle.json`;
2. an **access token** starting with `KGAT_` — read from `~/.kaggle/access_token` or the `KAGGLE_API_TOKEN` environment variable. This is what *Create New API Token* gives you on the current Kaggle settings page.

This cell gets one of them into place, trying four sources in order:

1. **An already present credential** (nothing to do);
2. **`ACCESS_TOKEN`** — paste your `KGAT_...` token between the quotes at the top of the cell. The cell writes it to `~/.kaggle/access_token` with mode `600` and exports `KAGGLE_API_TOKEN`;
3. **Colab Secrets** — the key icon in the left sidebar; add either `KAGGLE_USERNAME` + `KAGGLE_KEY`, or a single `KAGGLE_API_TOKEN`. Nothing is typed into the notebook;
4. **An upload picker** — choose either `kaggle.json` or a plain text file containing the `KGAT_` token; the cell detects which by trying to parse it as JSON.

Everything is inside `try/except`, and `KAGGLE_READY` records what happened. A missing token is **not** an error here: the next step's CC0 path trains a real model with no credentials at all, which is why this cell reports and continues instead of stopping the run. Tokens are written with mode `600`, because the Kaggle CLI warns about a world-readable token.


In [ ]:
# Step c (3/11) — put a Kaggle credential where the Kaggle CLI looks for it, if you have one.
import json
import os
from pathlib import Path

# Kaggle hands out two shapes of credential and the CLI reads both:
#   kaggle.json   {"username": ..., "key": ...}  -> ~/.kaggle/kaggle.json
#   access token  "KGAT_..."                     -> ~/.kaggle/access_token or $KAGGLE_API_TOKEN
KAGGLE_DIR = Path.home() / ".kaggle"
KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"
ACCESS_TOKEN_FILE = KAGGLE_DIR / "access_token"

# Paste the "KGAT_..." token between the quotes if you prefer this over Colab Secrets.
# Leave it empty to rely on the Secrets panel or the upload picker.
ACCESS_TOKEN = ""

KAGGLE_READY = (
    KAGGLE_JSON.is_file()
    or ACCESS_TOKEN_FILE.is_file()
    or bool(os.environ.get("KAGGLE_API_TOKEN"))
)

try:
    from google.colab import files as colab_files
    from google.colab import userdata as colab_userdata
except ImportError:  # running outside Colab
    colab_files = None
    colab_userdata = None


def store_json_token(username, key):
    """Write ~/.kaggle/kaggle.json the way the Kaggle CLI expects it."""
    if not username or not key:
        raise ValueError("both username and key are required")
    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    KAGGLE_JSON.write_text(json.dumps({"username": username, "key": key}), encoding="utf-8")
    KAGGLE_JSON.chmod(0o600)
    os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)


def store_access_token(token):
    """Write the newer KGAT_ access token, and export it for the CLI."""
    token = (token or "").strip()
    if not token:
        raise ValueError("empty access token")
    KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
    ACCESS_TOKEN_FILE.write_text(token + "\n", encoding="utf-8")
    ACCESS_TOKEN_FILE.chmod(0o600)
    os.environ["KAGGLE_API_TOKEN"] = token


if KAGGLE_READY:
    print("token         reusing an existing Kaggle credential")

if not KAGGLE_READY and ACCESS_TOKEN:
    store_access_token(ACCESS_TOKEN)
    KAGGLE_READY = True
    print(f"token         wrote {ACCESS_TOKEN_FILE} from ACCESS_TOKEN")

if not KAGGLE_READY and colab_userdata is not None:
    try:
        store_json_token(colab_userdata.get("KAGGLE_USERNAME"), colab_userdata.get("KAGGLE_KEY"))
        KAGGLE_READY = True
        print("token         read from Colab Secrets (KAGGLE_USERNAME / KAGGLE_KEY)")
    except Exception as exc:
        print(f"token         no Colab Secrets KAGGLE_USERNAME / KAGGLE_KEY ({type(exc).__name__})")
    if not KAGGLE_READY:
        try:
            store_access_token(colab_userdata.get("KAGGLE_API_TOKEN"))
            KAGGLE_READY = True
            print("token         read from Colab Secrets (KAGGLE_API_TOKEN)")
        except Exception as exc:
            print(f"token         no Colab Secrets KAGGLE_API_TOKEN ({type(exc).__name__})")

if not KAGGLE_READY and colab_files is not None:
    try:
        uploaded = colab_files.upload()  # pick kaggle.json, or a text file holding the KGAT_ token
        name, payload = next(iter(uploaded.items()))
        text = payload.decode("utf-8", "replace")
        try:
            document = json.loads(text)
        except json.JSONDecodeError:
            store_access_token(text)
            print(f"token         wrote {ACCESS_TOKEN_FILE} from {name}")
        else:
            store_json_token(document.get("username"), document.get("key"))
            print(f"token         wrote {KAGGLE_JSON} from {name}")
        KAGGLE_READY = True
    except Exception as exc:
        print(f"token         nothing uploaded ({type(exc).__name__})")

if KAGGLE_READY:
    print("kaggle        ready")
else:
    print("kaggle        NOT ready — set DATASET = 'cc0-bootstrap' in the next step, or add the token and re-run")


## d. Choose the training data

One switch, two paths that both lead to a trained model:

* **`"kaggle-catalogue"`** — `docs/adr/0003`: the Kaggle *Fashion Product Images* catalogue, 80 classes (`ml_workspace/artifacts/classes.json`), the eventual final training set. `prepare_data.py` downloads the archive, extracts `styles.csv` plus only the photographs whose label has `keep: true` in `ml_workspace/labels.json`, and writes `ml_workspace/data/{train,val,test}.csv` — rows of `id,image,label`, where `image` is relative to the catalogue root — plus `data/split_summary.json`. That download is ~2 GB and needs the token from step c; the extraction step keeps ~14 GB of JPEGs on the Colab disk.
* **`"cc0-bootstrap"`** — `alexeygrigorev/clothing-dataset-small` (CC0, ~103 MB `git clone`, 10 classes). No credentials. This is the set the app ships today, and it is how you try this whole notebook right now.

Both scripts fail loudly and explain themselves (missing token, network down, a `keep: true` label with fewer than 8 photographs), so the cell runs them through one `run()` helper that prints the exact command, streams the child's output, and turns a non-zero exit code into a short message. Every command that touched data or weights is appended to `COMMANDS`, which lands in `report.md` in step i.

In [ ]:
# Step d (4/11) — pick the training data.
DATASET = "kaggle-catalogue"   # "kaggle-catalogue" (needs the token above) | "cc0-bootstrap" (no credentials)

KAGGLE_DATASET = "paramaggarwal/fashion-product-images-dataset"
CC0_DIR = REPO / "ml_workspace" / "bootstrap" / "data" / "clothing-dataset-small"
COMMANDS = []  # every command that touched data or weights, quoted into report.md


def run(cmd):
    """Run one repository script from the repository root; stop with its message on failure."""
    print("$ " + " ".join(cmd))
    try:
        subprocess.run(cmd, cwd=REPO, check=True)
    except subprocess.CalledProcessError as exc:
        raise SystemExit(f"exit code {exc.returncode} — read the message above") from None
    COMMANDS.append(" ".join(cmd))


if DATASET == "kaggle-catalogue":
    run([sys.executable, "ml_workspace/scripts/prepare_data.py", "--dataset", KAGGLE_DATASET])
    CATALOGUE_ROOT = REPO / "ml_workspace" / "data" / "download" / "catalogue"
    for split in ("train", "val", "test"):
        manifest = REPO / "ml_workspace" / "data" / f"{split}.csv"
        if not manifest.is_file():
            raise SystemExit(f"{manifest} is missing — prepare_data.py did not finish")
    print(f"catalogue     {CATALOGUE_ROOT}")
elif DATASET == "cc0-bootstrap":
    run([sys.executable, "ml_workspace/bootstrap/fetch_dataset.py", "--dest", str(CC0_DIR)])
    print(f"clone         {CC0_DIR}")
else:
    raise SystemExit("DATASET must be 'kaggle-catalogue' or 'cc0-bootstrap'")

print(f"dataset       {DATASET}")

## e. Build the ultralytics ImageFolder out of symlinks

For classification, ultralytics takes `data=<root>` and expects an ImageFolder layout: `root/train/<Class>/*.jpg`, `root/val/<Class>/*.jpg`, and optionally `root/test/<Class>/*.jpg`. Each **folder name is the class name**, and ultralytics derives the class list by sorting those folder names — which is why the order in step i's `classes.json` is alphabetical and not the order of `labels.json`.

Our data is not in that shape (the catalogue is `styles.csv` + one flat `images/` directory, the CC0 clone uses `train/`, `validation/`, `test/` with lowercase folder names), and copying every photograph would duplicate ~14 GB on a disk Colab gives you for free-but-not-that-free. So we build the tree from **symlinks**: same layout, no second copy of the bytes.

The folder name is the label from `labels.json` (`T-Shirt`, `Casual Shoes`, …), so `model.names` comes out in a vocabulary the app knows. Two things are checked here rather than three steps later:

* every class has a `labels.json` entry for this dataset — a class with no Category or no Indonesian name is exactly what `gen_dart_labels.py` refuses to generate, and it must never reach the app;
* every class has enough photographs in train, val and test to be trainable and measurable.

For the CC0 path the lowercase folder names are translated through `ml_workspace/bootstrap/common.py`'s own `vocabulary()`/`dataset_classes()` helpers — reused, not copied — so `t-shirt` becomes `T-Shirt` the same way the bootstrap run does it.

Re-running the cell reuses the tree it already built (a `.build.json` marker records which dataset it came from) unless you delete `ml_workspace/yolo/dataset/`.

In [ ]:
# Step e (5/11) — build the ImageFolder tree ultralytics reads, out of symlinks.
import csv
import json
import os
import shutil

SPLITS = ("train", "val", "test")
DATASET_KEYS = {
    "kaggle-catalogue": "paramaggarwal/fashion-product-images-dataset",
    "cc0-bootstrap": "alexeygrigorev/clothing-dataset",
}
BUILD_MARKER = DATASET_DIR / ".build.json"

LABELS_DOC = json.loads((REPO / "ml_workspace" / "labels.json").read_text(encoding="utf-8"))
VOCAB = {entry["label"]: entry for entry in LABELS_DOC["labels"] if entry["dataset"] == DATASET_KEYS[DATASET]}


def link(source, target):
    """One symlink. Never a copy: 14 GB of JPEGs must not be duplicated."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists() or target.is_symlink():
        return
    os.symlink(Path(source).resolve(), target)


def build_plan():
    """(split, label, photograph) for every image we train on, read from our own data layout."""
    plan = []
    if DATASET == "kaggle-catalogue":
        for split in SPLITS:
            with (REPO / "ml_workspace" / "data" / f"{split}.csv").open(encoding="utf-8") as handle:
                for row in csv.DictReader(handle):
                    plan.append((split, row["label"], CATALOGUE_ROOT / row["image"]))
    else:
        sys.path.insert(0, str(REPO / "ml_workspace" / "bootstrap"))
        import common as bootstrap_common

        vocab = bootstrap_common.vocabulary(REPO / "ml_workspace" / "labels.json")
        pairs = bootstrap_common.dataset_classes(CC0_DIR, vocab)  # [(folder, label)], keep:true only
        for source_split, target_split in (("train", "train"), ("validation", "val"), ("test", "test")):
            for folder, label in pairs:
                for photo in sorted((CC0_DIR / source_split / folder).iterdir()):
                    plan.append((target_split, label, photo))
    return plan


MARKER = {}
if BUILD_MARKER.is_file():
    try:
        MARKER = json.loads(BUILD_MARKER.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        MARKER = {}

if MARKER.get("dataset") == DATASET:
    print(f"reusing {DATASET_DIR} (delete it to rebuild)")
else:
    shutil.rmtree(DATASET_DIR, ignore_errors=True)
    plan = build_plan()
    for split, label, photo in plan:
        link(photo, DATASET_DIR / split / label / Path(photo).name)
    BUILD_MARKER.write_text(json.dumps({"dataset": DATASET, "images": len(plan)}, indent=2), encoding="utf-8")
    print(f"linked {len(plan)} photographs into {DATASET_DIR} (no bytes copied)")

CLASSES = sorted(entry.name for entry in (DATASET_DIR / "train").iterdir() if entry.is_dir())
missing = [label for label in CLASSES if label not in VOCAB]
if missing:
    raise SystemExit(
        f"these classes have no labels.json entry for {DATASET_KEYS[DATASET]}: "
        + ", ".join(missing)
        + " — add {label, category, name, keep, dataset} for each, then re-run"
    )

COUNTS = {
    label: {split: sum(1 for _ in (DATASET_DIR / split / label).iterdir()) for split in SPLITS}
    for label in CLASSES
}
print(f"{'class':<22}{'category':<12}{'name':<24}{'train':>7}{'val':>6}{'test':>6}")
for label in CLASSES:
    entry = VOCAB[label]
    print(
        f"{label:<22}{entry['category']:<12}{entry['name']:<24}"
        f"{COUNTS[label]['train']:>7}{COUNTS[label]['val']:>6}{COUNTS[label]['test']:>6}"
    )
print(f"{len(CLASSES)} classes")

thin = {label: counts for label, counts in COUNTS.items() if counts["train"] < 8 or counts["val"] < 1 or counts["test"] < 1}
if thin:
    raise SystemExit(f"these classes are too thin to train and validate honestly: {thin}")

## f. Train

`yolov8n-cls.pt` is the **nano classification** checkpoint: ~5 MB, ImageNet-pretrained, and it fine-tunes to a new class list in minutes on 3k images and roughly 1.5–3 hours for the 80-class catalogue on a free T4. It is classification, not detection — no boxes, no NMS — so the exported graph has exactly one image input and one probability vector output.

The knobs worth knowing, all variables at the top of the cell:

* `EPOCHS` (default 30), `BATCH` (64 fits a 224 px T4; lower it if you get CUDA out-of-memory), `SEED` (42), `IMGSZ` (224 — the app's contract);
* `patience=10`: stop early when validation stops improving, instead of burning the default 100 epochs;
* `project=RUNS_DIR`, `name=RUN_NAME`: results land in `ml_workspace/yolo/runs/yolov8n-cls-224/`, not in `/content/runs/`, so the run directory travels with the repository;
* `workers=2`, because a free Colab VM has two vCPUs; more workers only add scheduling overhead.

ultralytics writes `weights/best.pt` (the epoch with the best validation top-1) and `weights/last.pt`; every step after this one uses `best.pt`. The last assertion is the one that matters for the app: the model's class order must equal the sorted folder names we built in step e, because `classes.json` in step i records exactly that order as the meaning of each output index.

In [ ]:
# Step f (6/11) — train yolov8n-cls at 224 px.
from pathlib import Path

from ultralytics import YOLO

MODEL = "yolov8n-cls.pt"   # try "yolo11n-cls.pt" or "yolov8s-cls.pt" for a bigger backbone
EPOCHS = 30
BATCH = 64                 # fits a T4 at 224 px; lower it on CUDA out-of-memory
SEED = 42
IMGSZ = 224                # the app's contract
PATIENCE = 10              # stop early when validation stops improving
RUN_NAME = "yolov8n-cls-224"

model = YOLO(MODEL)
final_metrics = model.train(
    data=str(DATASET_DIR),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    seed=SEED,
    patience=PATIENCE,
    device=DEVICE,
    workers=2,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,   # re-running overwrites this run instead of creating run2, run3 ...
    pretrained=True,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
CLASS_NAMES = [model.names[index] for index in range(len(model.names))]
COMMANDS.append(
    f"YOLO('{MODEL}').train(data={DATASET_DIR}, epochs={EPOCHS}, imgsz={IMGSZ}, "
    f"batch={BATCH}, seed={SEED}, project={RUNS_DIR})"
)

print(f"run           {RUN_DIR}")
print(f"best weights  {BEST_WEIGHTS} ({BEST_WEIGHTS.stat().st_size / 1e6:.1f} MB)")
print(f"last metrics  {final_metrics}")
print(f"{len(CLASS_NAMES)} classes in the model's order: {CLASS_NAMES}")
if CLASS_NAMES != CLASSES:
    raise SystemExit("the model's class order differs from the dataset folders — do not ship this model")

## g. Validate

`split="test"` is the held-out split — the ~10% no model trained on **and** never used to pick the best epoch — so it is the only number here that is not quietly optimistic. `metrics.top1` / `metrics.top5` are ultralytics' own top-1/top-5 accuracy; `metrics.confusion_matrix` is a `[N,N]` matrix indexed `[predicted][true]`, which is what the per-class table is computed from — recall for a class is `matrix[i,i] / matrix[:,i].sum()`.

Everything is written to `ml_workspace/yolo/artifacts/` so it can be quoted in `report.md` (step i) and checked by a human later:

* `per_class_accuracy.csv` — label, support, correct, recall;
* `confusion_matrix.csv` — the raw counts, so a disagreement can be looked up without re-running anything;
* `confusion_matrix.png` and `confusion_matrix_normalized.png` — ultralytics' plots of the same matrix.

A glance at the per-class table is worth more than the single top-1 number: with 80 classes, a healthy top-1 can still hide two classes that are always confused with each other, and that is what the confusion matrix shows.

In [ ]:
# Step g (7/11) — measure the trained model on a held-out split.
import csv as csv_module
import shutil as shutil_module

import numpy as np
from ultralytics import YOLO

SPLIT = "test"   # "val" measures the split that picked the best epoch

trained = YOLO(str(BEST_WEIGHTS))
metrics = trained.val(
    data=str(DATASET_DIR),
    split=SPLIT,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    plots=True,
    project=str(RUNS_DIR),
    name=f"val-{SPLIT}",
    exist_ok=True,
)
TOP1 = float(metrics.top1)
TOP5 = float(metrics.top5)
print(f"top-1         {TOP1:.4f}")
print(f"top-5         {TOP5:.4f}")

confusion = metrics.confusion_matrix
CONFUSION_NAMES = [confusion.names[index] for index in range(confusion.nc)]
CONFUSION_MATRIX = np.asarray(confusion.matrix)  # matrix[predicted][true]
SUPPORT = CONFUSION_MATRIX.sum(axis=0)
CORRECT = np.diag(CONFUSION_MATRIX)
RECALL = {name: float(CORRECT[index] / max(1.0, SUPPORT[index])) for index, name in enumerate(CONFUSION_NAMES)}

with (ARTIFACTS_DIR / "per_class_accuracy.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv_module.writer(handle)
    writer.writerow(("label", "support", "correct", "recall"))
    for index, name in enumerate(CONFUSION_NAMES):
        writer.writerow((name, int(SUPPORT[index]), int(CORRECT[index]), f"{RECALL[name]:.4f}"))

with (ARTIFACTS_DIR / "confusion_matrix.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv_module.writer(handle)
    writer.writerow(("predicted \\ true", *CONFUSION_NAMES))
    for index, name in enumerate(CONFUSION_NAMES):
        writer.writerow((name, *[int(value) for value in CONFUSION_MATRIX[index]]))

for png in sorted(Path(metrics.save_dir).glob("confusion_matrix*.png")):
    shutil_module.copy2(png, ARTIFACTS_DIR / png.name)
    print(f"plot          {ARTIFACTS_DIR / png.name}")

print(f"{'class':<22}{'support':>8}{'correct':>9}{'recall':>9}")
for index, name in enumerate(CONFUSION_NAMES):
    print(f"{name:<22}{int(SUPPORT[index]):>8}{int(CORRECT[index]):>9}{RECALL[name]:>9.4f}")

COMMANDS.append(f"YOLO('{BEST_WEIGHTS}').val(data={DATASET_DIR}, split='{SPLIT}', imgsz={IMGSZ})")

## h. Export to `.tflite`, then verify the contract and the conversion

**Part 1** exports the graph: `format="litert"` is the current name for the LiteRT export and produces a `.tflite`; `format="tflite"` still works but is deprecated and just forwards here with a warning. For classification there is no NMS or box decoding in the graph. The export runs on CPU — tracing does not need the GPU, and the small model converts in well under a minute.

**Part 2** is the part that decides whether the file is shippable, and it is deliberately paranoid, because a wrong model that *loads* is worse than one that fails:

* An **independent runtime** reads the file — `tf.lite.Interpreter` on Colab, `ai_edge_litert` otherwise — never the exporter that wrote it. Step h's assertions are exactly what `scan_pipeline.dart::checkContracts` enforces at load: float32 input, batch 1, `[1,224,224,3]` **or** `[1,3,224,224]` (the app reads the shape and packs accordingly — the cell prints which layout your export has), `[1,N]` float32 output with `N` = the class count, and a **probability distribution**.
* If the output is not a distribution (logits), the cell re-exports with an explicit `nn.Softmax` on the end and re-checks. In ultralytics 8.4.177 the classification head already ends in `Softmax`, so this fallback is expected *not* to trigger — it is there because "expected" is not "verified", and a silent logits output would make the app's confidence bar meaningless rather than wrong-looking.
* Then the **fidelity gate**: ≥100 held-out photographs go through both the PyTorch model and the `.tflite`, and we print how many argmaxes agree. Both sides get **the same pixels** (short-side resize + centre crop, `[0,1]` RGB) so the number measures the conversion, not two different preprocessing pipelines. The gate is the one `ml_workspace/bootstrap/export_tflite.py tflite` applies: `.tflite` accuracy ≥ 0.50 and within 0.10 absolute of PyTorch's — a broken conversion cannot reach the zip.

In [ ]:
# Step h, part 1 (8/11) — export the trained model to LiteRT (.tflite).
import hashlib

from ultralytics import YOLO

TFLITE_PATH = ARTIFACTS_DIR / "garment_classifier.tflite"
EXPORT_STEPS = []

trained = YOLO(str(BEST_WEIGHTS))
exported_path = Path(trained.export(format="litert", imgsz=IMGSZ, device="cpu"))
shutil.copy2(exported_path, TFLITE_PATH)
EXPORT_STEPS.append(f"YOLO('{BEST_WEIGHTS}').export(format='litert', imgsz={IMGSZ}, device='cpu')")

print(f"exported      {exported_path}")
print(f"copied to     {TFLITE_PATH} ({TFLITE_PATH.stat().st_size / 1e6:.2f} MB)")
print(f"sha256        {hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()}")

In [ ]:
# Step h, part 2 (9/11) — verify the export against the app's frozen contract, then against PyTorch.
import importlib.util
import math
import shutil
from collections import namedtuple

import numpy as np
import torch
from PIL import Image

Contract = namedtuple(
    "Contract", "runtime interpreter in_index in_shape in_dtype out_index out_shape out_dtype layout"
)


def load_interpreter(path):
    """A TFLite runtime that is independent of the exporter that wrote the file."""
    if importlib.util.find_spec("tensorflow"):
        import tensorflow as tf

        return tf.lite.Interpreter(model_path=str(path)), f"tensorflow {tf.__version__} tf.lite.Interpreter"
    from ai_edge_litert.interpreter import Interpreter

    return Interpreter(model_path=str(path)), "ai_edge_litert.interpreter.Interpreter"


def inspect_model(path):
    """Read the tensor IO the app will look at, and which of the two layouts it is."""
    interpreter, runtime = load_interpreter(path)
    interpreter.allocate_tensors()
    inputs = interpreter.get_input_details()[0]
    outputs = interpreter.get_output_details()[0]
    shape = tuple(int(value) for value in inputs["shape"])
    if shape[1:] == (IMGSZ, IMGSZ, 3):
        layout = "NHWC"
    elif shape[1:] == (3, IMGSZ, IMGSZ):
        layout = "NCHW"
    else:
        raise SystemExit(f"unexpected input shape {shape}: the app packs [1,224,224,3] or [1,3,224,224]")
    return Contract(
        runtime,
        interpreter,
        inputs["index"],
        shape,
        inputs["dtype"],
        outputs["index"],
        tuple(int(value) for value in outputs["shape"]),
        outputs["dtype"],
        layout,
    )


def assert_app_contract(contract, class_count):
    """Exactly what flutter_app/lib/logic/scan_pipeline.dart refuses to load."""
    assert contract.in_dtype == np.float32, f"input dtype {contract.in_dtype.__name__}: the app feeds float32"
    assert contract.out_dtype == np.float32, f"output dtype {contract.out_dtype.__name__}: the app reads float32"
    assert contract.in_shape[0] == 1 and contract.out_shape[0] == 1, "both tensors must have batch 1"
    assert contract.out_shape[1] == class_count, (
        f"the model scores {contract.out_shape[1]} classes but classes.json will hold {class_count}: "
        "the app throws an Indonesian error naming exactly this mismatch"
    )


def predict(contract, pixels_hwc):
    """[H,W,3] float32 RGB in [0,1] -> the model's [N] float32 output."""
    array = pixels_hwc[None] if contract.layout == "NHWC" else np.transpose(pixels_hwc, (2, 0, 1))[None]
    contract.interpreter.set_tensor(contract.in_index, array.astype(np.float32))
    contract.interpreter.run()
    return contract.interpreter.get_tensor(contract.out_index)[0]


def load_pixels(path, size=IMGSZ):
    """Identical pixels for both models: RGB, short side resized, centre cropped, [0,1] float32.

    This is ultralytics' own evaluation pipeline (Resize -> CenterCrop -> ToTensor, mean 0 / std 1).
    """
    image = Image.open(path).convert("RGB")
    scale = size / min(image.size)
    image = image.resize(
        (max(size, round(image.width * scale)), max(size, round(image.height * scale))), Image.BILINEAR
    )
    left, top = (image.width - size) // 2, (image.height - size) // 2
    return np.asarray(image.crop((left, top, left + size, top + size)), dtype=np.float32) / 255.0


network = trained.model.eval()  # eval mode: the classification head returns probabilities, not logits


def torch_probabilities(pixels_hwc):
    """The trained PyTorch network on the very same pixels — the reference the export must agree with."""
    with torch.no_grad():
        output = network(torch.from_numpy(np.transpose(pixels_hwc, (2, 0, 1)))[None])
        probabilities = output[0] if isinstance(output, (tuple, list)) else output
        return probabilities[0].detach().cpu().numpy()


PROBE = load_pixels(sorted((DATASET_DIR / SPLIT / CLASSES[0]).iterdir())[0])

contract = inspect_model(TFLITE_PATH)
print(f"runtime       {contract.runtime}")
print(f"input         {contract.in_shape} {contract.in_dtype.__name__}")
print(f"output        {contract.out_shape} {contract.out_dtype.__name__}")
print(f"layout        {contract.layout} (the app reads the tensor shape and packs either)")
assert_app_contract(contract, len(CLASSES))

PROBE_SUM = float(predict(contract, PROBE).sum())
print(f"sum of scores {PROBE_SUM:.6f}")

if not 0.99 <= PROBE_SUM <= 1.01:
    print("the graph returns logits, not probabilities — re-exporting with an explicit Softmax on the end")

    for module in network.modules():
        if type(module).__name__ == "Classify":
            module.export = False  # in eval mode the head then returns (probabilities, logits): we want the logits


    class WithSoftmax(torch.nn.Module):
        """[1,3,224,224] RGB in [0,1] -> [1,N] softmax probabilities, which is the app's contract."""

        def __init__(self, network):
            super().__init__()
            self.network = network

        def forward(self, x):
            output = self.network(x)
            logits = output[1] if isinstance(output, (tuple, list)) else output
            return torch.nn.functional.softmax(logits, dim=1)


    from ultralytics.utils.export.litert import torch2litert

    target = TFLITE_PATH.with_name("garment_classifier_softmax.tflite")
    torch2litert(WithSoftmax(network), torch.zeros(1, 3, IMGSZ, IMGSZ), target, metadata={"task": "classify"})
    shutil.move(str(target), str(TFLITE_PATH))
    EXPORT_STEPS.append("torch2litert(WithSoftmax(best.pt), torch.zeros(1,3,224,224), ...) -> garment_classifier.tflite")

    contract = inspect_model(TFLITE_PATH)
    assert_app_contract(contract, len(CLASSES))
    PROBE_SUM = float(predict(contract, PROBE).sum())
    print(f"sum of scores {PROBE_SUM:.6f} (after the re-export)")
    if not 0.99 <= PROBE_SUM <= 1.01:
        raise SystemExit("the export is still not a probability distribution — do not ship it")

# Fidelity gate: >= 100 held-out photographs, same pixels into both models, compare the argmaxes.
PER_CLASS = max(2, math.ceil(120 / len(CLASSES)))
held_out = []
for label in CLASSES:
    photos = sorted((DATASET_DIR / SPLIT / label).iterdir())
    step = max(1, len(photos) // PER_CLASS)
    held_out.extend((photo, label) for photo in photos[::step][:PER_CLASS])
if len(held_out) < 100:
    raise SystemExit(f"only {len(held_out)} held-out photographs — the fidelity check needs at least 100")

rows = []
for photo, label in held_out:
    pixels = load_pixels(photo)
    expected = torch_probabilities(pixels)
    actual = predict(contract, pixels)
    rows.append(
        (
            label,
            CLASSES[int(expected.argmax())],
            CLASSES[int(actual.argmax())],
            float(actual.sum()),
            float(np.abs(expected - actual).max()),
        )
    )

IMAGE_COUNT = len(rows)
AGREE_COUNT = sum(1 for _, reference, converted, _, _ in rows if reference == converted)
TORCH_CORRECT = sum(1 for label, reference, _, _, _ in rows if label == reference)
TFLITE_CORRECT = sum(1 for label, _, converted, _, _ in rows if label == converted)
AGREEMENT = AGREE_COUNT / IMAGE_COUNT
TORCH_ACCURACY = TORCH_CORRECT / IMAGE_COUNT
TFLITE_ACCURACY = TFLITE_CORRECT / IMAGE_COUNT
MAX_SUM_ERROR = max(abs(row[3] - 1.0) for row in rows)
MAX_PROB_DIFF = max(row[4] for row in rows)
GATE_OK = TFLITE_ACCURACY >= 0.50 and TFLITE_ACCURACY >= TORCH_ACCURACY - 0.10

print(f"images        {IMAGE_COUNT} held out of training ({SPLIT} split)")
print(f"accuracy      pytorch {TORCH_ACCURACY:.4f}   .tflite {TFLITE_ACCURACY:.4f}")
print(f"agreement     {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} argmaxes identical")
print(f"max |sum-1|   {MAX_SUM_ERROR:.3e}")
print(f"max |p diff|  {MAX_PROB_DIFF:.4f}")
print(f"gate          .tflite >= 0.50 and >= pytorch - 0.10 -> {'PASSED' if GATE_OK else 'FAILED'}")
if not GATE_OK:
    raise SystemExit(
        f"converted model scored {TFLITE_ACCURACY:.4f} against PyTorch's {TORCH_ACCURACY:.4f} "
        "on the same held-out images: the conversion is broken, do not ship it"
    )

## i. Write `classes.json` and `report.md`

`classes.json` is this repository's format — `{"version": 1, "source": ..., "labels": [...]}` — and the order of `labels` **is** the model's output order. Since step e named the class folders with the `labels.json` spellings, ultralytics sorted them alphabetically, and `model.names` is that sorted order: index `i` of `labels` is what score `i` means. `scripts/gen_dart_labels.py` reads this file together with `ml_workspace/labels.json` and generates `flutter_app/lib/logic/garment_labels.dart` from it, so this order is not cosmetic — reorder it by hand and every prediction in the app is off by one class.

`report.md` is the evidence trail: the environment, the exact commands from steps d–h, the measured numbers (top-1/top-5, the conversion gate, per-class recall), the tensor IO as read by the verifier, the `.tflite`'s sha256 and byte size, and the licence note. Its numbers come from variables this notebook just measured, so a re-run rewrites it with its own numbers — never hand-edit it.

In [ ]:
# Step i (10/11) — write the class list and the report next to the model.
import json
from datetime import datetime, timezone

CLASSES_JSON = ARTIFACTS_DIR / "classes.json"
CLASSES_JSON.write_text(
    json.dumps(
        {
            "version": 1,
            "source": (
                f"Ultralytics {MODEL} classification fine-tuned on {DATASET} "
                f"({len(CLASSES)} classes, imgsz {IMGSZ}, {EPOCHS} epochs, batch {BATCH}, seed {SEED}); "
                "index = the model's output index (ultralytics' alphabetical class-folder order)"
            ),
            "labels": CLASSES,
        },
        indent=2,
        ensure_ascii=False,
    )
    + "\n",
    encoding="utf-8",
)
print(f"wrote         {CLASSES_JSON} ({len(CLASSES)} labels)")

TFLITE_BYTES = TFLITE_PATH.stat().st_size
TFLITE_SHA256 = hashlib.sha256(TFLITE_PATH.read_bytes()).hexdigest()

per_class_table = "\n".join(
    f"| {name} | {VOCAB[name]['category']} | {VOCAB[name]['name']} | "
    f"{int(SUPPORT[index])} | {int(CORRECT[index])} | {RECALL[name]:.4f} |"
    for index, name in enumerate(CONFUSION_NAMES)
)
class_list = ", ".join(CLASSES)
command_lines = "\n".join(COMMANDS + EXPORT_STEPS)
# Hoisted out of the report's f-string: a replacement field cannot carry quotes of the same
# type as the f-string itself on older Pythons, and Colab's Python version is not ours to pick.
DATE_STAMP = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M")
GATE_LABEL = "PASSED" if GATE_OK else "FAILED"
GPU_NAME = torch.cuda.get_device_name(0) if GPU else "none (CPU)"
SIZES = {split: sum(counts[split] for counts in COUNTS.values()) for split in SPLITS}

report = f"""# YOLOv8 classifier run — {DATASET}

| | |
| --- | --- |
| date (UTC) | {DATE_STAMP} |
| repo commit | `{GIT_SHA}` |
| GPU | {GPU_NAME} |
| ultralytics / torch / python | {ultralytics.__version__} / {torch.__version__} / {sys.version.split()[0]} |
| model | `{MODEL}` (ImageNet-pretrained), fine-tuned head |
| data | `{DATASET}` — {len(CLASSES)} classes, {SIZES['train']} train / {SIZES['val']} val / {SIZES['test']} test images |
| training | imgsz {IMGSZ}, {EPOCHS} epochs, batch {BATCH}, seed {SEED}, patience {PATIENCE}, device {DEVICE} |
| best weights | `{BEST_WEIGHTS}` |

## Measured

| metric | value |
| --- | --- |
| top-1 (`{SPLIT}` split, ultralytics pipeline) | {TOP1:.4f} |
| top-5 (`{SPLIT}` split, ultralytics pipeline) | {TOP5:.4f} |
| PyTorch top-1 on the {IMAGE_COUNT}-image fidelity subset | {TORCH_ACCURACY:.4f} |
| `.tflite` top-1 on the same subset | {TFLITE_ACCURACY:.4f} |
| argmax agreement, `.tflite` vs PyTorch | {AGREE_COUNT}/{IMAGE_COUNT} = {AGREEMENT:.4f} |
| largest `abs(sum(scores) - 1)` | {MAX_SUM_ERROR:.3e} |
| largest `abs(p_tflite - p_pytorch)` | {MAX_PROB_DIFF:.4f} |
| conversion gate (`.tflite` >= 0.50 and >= PyTorch - 0.10) | {GATE_LABEL} |

The fidelity subset is {IMAGE_COUNT} `{SPLIT}`-split photographs, up to {PER_CLASS} per class. Both models
are fed the *same* pixels (RGB, short side resized to {IMGSZ}, centre cropped, `[0,1]` float32), so the
agreement number isolates the conversion rather than two different preprocessing pipelines. The
top-1/top-5 above come from ultralytics' own evaluation on the whole `{SPLIT}` split.

## Exported model

* `{TFLITE_PATH}`
* size {TFLITE_BYTES:,} bytes, sha256 `{TFLITE_SHA256}`
* read by `{contract.runtime}`: input `{contract.in_shape}` {contract.in_dtype.__name__}, output `{contract.out_shape}` {contract.out_dtype.__name__}, layout {contract.layout}
* class order (`classes.json` `labels`, = the model's output indices): {class_list}

## Commands

```bash
pip install -q ultralytics kaggle
git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
{command_lines}
```

## Per class (`{SPLIT}` split)

| class | Category | Indonesian name | support | correct | recall |
| --- | --- | --- | --- | --- | --- |
{per_class_table}

## Licence

Trained with Ultralytics YOLOv8; Ultralytics' licence page states that models trained with their
code are AGPL-3.0 by default, and these weights are derivative work of the `{MODEL}` checkpoint.
See `docs/adr/0006-yolov8-classifier-agpl.md`: shipping a build that contains this `.tflite`
triggers AGPL-3.0's source obligation, and this repository has no LICENSE file.
"""
(ARTIFACTS_DIR / "report.md").write_text(report, encoding="utf-8")
print(f"wrote         {ARTIFACTS_DIR / 'report.md'}")

## j. Zip and download

Colab's VM disappears when the session ends, so the last step packs the shippable set into one archive and hands it to your browser:

* `garment_classifier.tflite` — the model, with its sha256 and byte size printed below;
* `classes.json` — the class order that model was exported with;
* `report.md` — the measured numbers and the exact commands;
* `confusion_matrix.png` / `confusion_matrix_normalized.png` — for the report and the presentation;
* `per_class_accuracy.csv` / `confusion_matrix.csv` — the same numbers as text.

Where each file goes in the repository, and every command that follows, is the *Integration steps* half of `ml_workspace/yolo/README.md`. Copy the sha256 line into `flutter_app/assets/models/LICENSES.md` when you land the model — that file is where the app records what it ships.

In [ ]:
# Step j (11/11) — pack the artifacts and download them.
import zipfile

ZIP_PATH = ARTIFACTS_DIR / "gayaku_yolo_classifier.zip"
BUNDLE = (
    "garment_classifier.tflite",
    "classes.json",
    "report.md",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "per_class_accuracy.csv",
    "confusion_matrix.csv",
)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in BUNDLE:
        path = ARTIFACTS_DIR / name
        if path.is_file():
            archive.write(path, name)
        else:
            print(f"missing, skipped: {name}")

print(f"{TFLITE_PATH.name:<32}{TFLITE_BYTES:>12,} bytes")
print(f"{'sha256':<32}{TFLITE_SHA256}")
print(f"{ZIP_PATH.name:<32}{ZIP_PATH.stat().st_size:>12,} bytes")
print("next: ml_workspace/yolo/README.md — Integration steps")

try:
    from google.colab import files as colab_files

    colab_files.download(str(ZIP_PATH))
except Exception as exc:
    print(f"download skipped ({type(exc).__name__}); the zip is at {ZIP_PATH}")

## k. The licence consequence of this choice

Ultralytics' own licence page states that **all models trained with Ultralytics code are AGPL-3.0 by default**, and the exported weights are derivative work of the `yolov8n-cls.pt` checkpoint — so the `.tflite` you just produced carries AGPL-3.0, no matter that we trained it ourselves on our own data. `docs/adr/0006-yolov8-classifier-agpl.md` records this decision and its consequences; `docs/adr/0002-permissive-model-licences.md` is the rule it supersedes for the garment classifier (that rule still governs the matting model).

Concretely:

* **Coursework, research, a private build:** unaffected. Ultralytics lists university coursework as free AGPL use; the obligation begins at *distribution*.
* **Distributing the APK** (Play Store, a friend's phone via a file, a public release) means you are distributing the model as part of your app: AGPL-3.0 requires the complete corresponding source of that derivative work to be offered under AGPL-3.0. **This repository has no `LICENSE` file**, so adding one is now a prerequisite for distributing a build that contains this model — not for training it.
* The Flutter plugin `ultralytics_yolo` is **not** part of this path and must not be added: the app bundles the exported `.tflite` and runs it through `tflite_flutter`, which is Apache-2.0. The licence attaches to the training code and the weights, not to our inference glue.
* Using a *published* YOLO weight set instead of training one does not escape this: Ultralytics' position covers models produced with their training code too, and the popular fashion YOLO checkpoints are AGPL-3.0 upstream (see ADR-0006).
* The training data is a separate question: the Kaggle catalogue's MIT tag covers its labels, not
  the photographs, and `docs/adr/0003-kaggle-catalogue-training-data.md` already keeps those
  weights coursework-only. The CC0 bootstrap clone is CC0.